# LIDS Demo

This notebook demonstrates the similarity-metric portion of **LIDS (LLM Summary Inference Under the Layered Lens)**:

1. Convert text into BERT token embeddings.
2. Compute the SVD of the token-embedding matrix.
3. Construct the LIDS direction vectors across the leading SVD layers.
4. Compare texts using the maximum absolute cosine similarity over aligned layers.
5. Visualize the resulting pairwise LIDS similarity matrix.

The example is self-contained: it uses a small set of example paragraphs so the notebook can be run immediately without requiring the research dataset. Replace the example texts with your own paragraphs when desired.

## 1. Install dependencies

From the repository root, install the dependencies with:

```bash
python -m pip install -r requirements.txt
```

The notebook itself assumes the packages in `requirements.txt` are already installed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os

from transformers import BertTokenizer, BertModel

# Sets up the correct directory to pull the functions properly.
os.chdir("..")
print(os.getcwd())

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Load BERT

LIDS begins with the token-level BERT representation. `bert-base-uncased` produces 768-dimensional token embeddings, so each text is represented by an embedding matrix `X` with one row per retained token.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.to(device)
model.eval()

print("BERT loaded successfully.")
print(f"Hidden dimension: {model.config.hidden_size}")

## 3. Example texts

The notebook uses the same three comparisons as the quick start: two gardening comparisons and one deliberately unrelated gardening/remote-work comparison.

In [ ]:
texts = [
    # 1. Reference text: longer reference
    "Successful gardening requires regular attention to several important factors. 
    Plants need healthy soil, sufficient sunlight, and consistent watering to grow properly throughout the season. 
    Gardeners must also monitor their plants, remove weeds, and prune when necessary to prevent problems and maintain healthy growth. 
    Adjusting care based on seasonal conditions can help keep a garden productive and healthy.",

    # 2. Summary: similar terminology and phrasing
    "Successful gardening requires healthy soil, sufficient sunlight, consistent watering, and regular maintenance such as removing weeds and pruning.",

    # 3. Summary: varying vocabulary, same information
    "A thriving garden depends on fertile ground, adequate light, reliable hydration, and routine upkeep to support plant growth throughout the year.",

    # 4. Completely unrelated
    "Researchers developed a new spacecraft navigation system that uses onboard sensors to calculate trajectories and make precise adjustments during long-distance missions."
]

text_names = ["reference", "garden summary similar vocabulary", "garden summary differing vocabulary", "unrelated to reference"]

for name, text in zip(text_names, texts):
    print(f"{name}: {text}")

## 4. Construct the LIDS direction vectors

For a text with embedding matrix `X`, compute

`X = U Σ Vᵀ`.

For each singular layer `l`, define the sign correction

`s_l = sign(<v_l, p^(-1/2) 1_p>)`,

and construct the cumulative direction vector

`d(k) = Σ_{l=1}^k σ_l^α s_l Xᵀ u_l`.

The implementation below stores `d(1), ..., d(k_max)` for each text. The default demonstration uses `alpha = 1` and `k_layers = 11`, matching the parameters used in the current LIDS experiments.

In [ ]:
def compute_direction_vectors(
    text,
    tokenizer,
    model,
    k_layers=11,
    alpha=1.0,
    max_length=512,
    device=None,
):
    """Compute cumulative LIDS direction vectors for one text."""
    if device is None:
        device = next(model.parameters()).device

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        add_special_tokens=True,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    # Last hidden state: (1, number_of_tokens, 768).
    embeddings = outputs.last_hidden_state[0]
    attention_mask = encoded["attention_mask"][0].bool()

    # Remove BERT special tokens from the embedding matrix.
    input_ids = encoded["input_ids"][0]
    special_mask = tokenizer.get_special_tokens_mask(
        input_ids.tolist(), already_has_special_tokens=True
    )
    special_mask = torch.tensor(special_mask, device=device, dtype=torch.bool)

    keep = attention_mask & ~special_mask
    X = embeddings[keep].detach().cpu().numpy().astype(np.float64)

    if X.shape[0] == 0:
        raise ValueError("The text produced no non-special BERT tokens.")

    # SVD: X = U diag(S) V^T.
    U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)
    V = Vt.T

    n_tokens, p = X.shape
    k_max = min(k_layers, n_tokens, p)

    direction_vectors = []
    direction = np.zeros(p, dtype=np.float64)
    ones_normalized = np.ones(p, dtype=np.float64) / np.sqrt(p)

    for l in range(k_max):
        v_l = V[:, l]
        u_l = U[:, l]
        sigma_l = singular_values[l]

        sign = np.sign(np.dot(v_l, ones_normalized))
        if sign == 0:
            sign = 1.0

        layer_direction = (
            (sigma_l ** alpha)
            * sign
            * (X.T @ u_l)
        )

        direction = direction + layer_direction
        direction_vectors.append(direction.copy())

    return direction_vectors, X, singular_values[:k_max]


## 5. Compute direction-vector sets

Each text now has a sequence of LIDS direction vectors, one for each retained SVD layer.

In [ ]:
k_layers = 11
alpha = 1.0
max_length = 512

direction_vector_sets = []
embedding_matrices = []
singular_value_sets = []

print("Computing direction vectors...\n")

for name, text in zip(text_names, texts):
    print(f"Processing {name}")

    direction_vectors, embedding_matrix, singular_values = compute_direction_vectors(
        text=text,
        tokenizer=tokenizer,
        model=model,
        k_layers=k_layers,
        alpha=alpha,
        max_length=max_length,
        device=device,
    )

    direction_vector_sets.append(direction_vectors)
    embedding_matrices.append(embedding_matrix)
    singular_value_sets.append(singular_values)

print("\nFinished computing direction vectors.")

In [ ]:
print("Direction-vector summary:\n")

for name, X, directions in zip(text_names, embedding_matrices, direction_vector_sets):
    print(
        f"{name}: embedding matrix {X.shape}, "
        f"{len(directions)} LIDS layers, "
        f"direction dimension {directions[0].shape[0]}"
    )

## 6. LIDS similarity

For two texts `a` and `b`, compare their direction vectors at aligned values of `k` using absolute cosine similarity:

`|CS(d_a(k), d_b(k))|`.

The LIDS similarity is the maximum over the aligned layers.

In [ ]:
def cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return float(np.dot(vec1, vec2) / (norm1 * norm2))


def compare_direction_vector_sets(
    direction_vectors_a,
    direction_vectors_b,
    absolute_value=True,
):
    """Compare two LIDS direction-vector sets across aligned layers.

    Returns:
        best_k: 1-based layer producing the maximum similarity.
        best_similarity: corresponding LIDS similarity.
        layer_similarities: similarity at every aligned layer.
    """
    num_layers = min(len(direction_vectors_a), len(direction_vectors_b))

    if num_layers == 0:
        raise ValueError("Both direction-vector sets must contain at least one layer.")

    layer_similarities = []

    for k in range(num_layers):
        similarity = cosine_similarity(
            direction_vectors_a[k],
            direction_vectors_b[k],
        )

        if absolute_value:
            similarity = abs(similarity)

        layer_similarities.append(similarity)

    best_index = int(np.argmax(layer_similarities))

    return (
        best_index + 1,
        layer_similarities[best_index],
        layer_similarities,
    )

In [ ]:
# Compare the reference with each of the other three texts.
for comparison_index in range(1, len(texts)):
    best_k, similarity, layer_similarities = compare_direction_vector_sets(
        direction_vector_sets[0],
        direction_vector_sets[comparison_index],
        absolute_value=True,
    )
    print(
        f"reference vs. {text_names[comparison_index]}: "
        f"similarity={similarity:.6f}, best k={best_k}"
    )

## 7. Pairwise LIDS similarity matrix

This is the main visual demonstration. Each entry is the LIDS similarity between two paragraphs. Values are based directly on the maximum absolute cosine similarity over the aligned direction-vector layers.

In [ ]:
n = len(texts)
similarity_matrix = np.zeros((n, n), dtype=np.float64)
best_k_matrix = np.zeros((n, n), dtype=int)

print("Computing pairwise LIDS similarities...\n")

for i in range(n):
    for j in range(n):
        best_k, similarity, _ = compare_direction_vector_sets(
            direction_vector_sets[i],
            direction_vector_sets[j],
            absolute_value=True,
        )

        similarity_matrix[i, j] = similarity
        best_k_matrix[i, j] = best_k

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=text_names,
    columns=text_names,
)

print("LIDS Similarity Matrix")
display(similarity_df.round(4))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    similarity_df,
    annot=True,
    fmt=".3f",
    square=True,
    vmin=0.0,
    vmax=1.0,
    cbar=True,
)
plt.title("LIDS Similarity Matrix")
plt.tight_layout()
plt.show()

## 8. Inspect the optimal layer for each pair

Because LIDS searches across the aligned latent SVD layers, it is useful to inspect which `k` produces the maximum similarity.

In [ ]:
best_k_df = pd.DataFrame(
    best_k_matrix,
    index=text_names,
    columns=text_names,
)

print("Optimal k for each pair")
display(best_k_df)

## 9. Simple sanity checks

The absolute cosine formulation gives similarities in `[0, 1]`, and a text compared with itself should have similarity 1.

In [ ]:
# Range check.
assert np.all(similarity_matrix >= -1e-12)
assert np.all(similarity_matrix <= 1.0 + 1e-12)

# Symmetry check.
assert np.allclose(similarity_matrix, similarity_matrix.T, atol=1e-10)

# Self-similarity check.
assert np.allclose(np.diag(similarity_matrix), 1.0, atol=1e-10)

print("All LIDS sanity checks passed.")

## 10. Save the similarity matrix

The matrix can be exported for later analysis.

In [ ]:
output_path = "lids_similarity_matrix.csv"
similarity_df.to_csv(output_path)
print(f"Saved similarity matrix to: {output_path}")

## What this demonstrates

The notebook displays the main computational structure of LIDS without requiring the reader to navigate the full research codebase:

**Text → BERT embeddings → SVD → layered direction vectors → maximum absolute cosine similarity → similarity matrix.**

For day-to-day use, see `LIDS_quickstart.ipynb` and import the reusable implementation from the `lids` package.